In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from link_prediction.config import (
    RESULTS_DIR,
)
from link_prediction.sensitivity import (
    run_parameter_sensitivity,
)

In [ ]:
benchmark_name = "revision"

figure_directory = (
    RESULTS_DIR
    / "figures"
    / benchmark_name
)

figure_directory.mkdir(
    parents=True,
    exist_ok=True,
)

(
    fold_metrics,
    network_summary,
    overall_summary,
) = run_parameter_sensitivity(
    benchmark_name=
        benchmark_name,
)

In [ ]:
overall_summary[
    [
        "method_id",
        "method",
        "parameter",
        "parameter_value",
        "is_primary",
        "network_count",
        "average_precision_mean",
        "average_precision_sd_across_networks",
        "roc_auc_mean",
        "roc_auc_sd_across_networks",
    ]
].sort_values(
    [
        "method_id",
        "parameter_value",
    ]
).reset_index(
    drop=True
)

In [ ]:
method_ids = [
    "lpi",
    "ora_cni",
    "lit",
    "lrw",
    "srw",
    "pfp",
]

figure, axes = plt.subplots(
    2,
    3,
    figsize=(
        15,
        8,
    ),
)

for method_id, axis in zip(
    method_ids,
    axes.ravel(),
    strict=True,
):
    method_summary = (
        overall_summary[
            overall_summary[
                "method_id"
            ]
            == method_id
        ]
        .sort_values(
            "parameter_value"
        )
    )

    axis.errorbar(
        method_summary[
            "parameter_value"
        ],
        method_summary[
            "average_precision_mean"
        ],
        yerr=method_summary[
            "average_precision_sd_across_networks"
        ],
        marker="o",
        capsize=4,
        color="#4472C4",
    )

    primary = method_summary[
        method_summary[
            "is_primary"
        ]
    ]

    axis.scatter(
        primary[
            "parameter_value"
        ],
        primary[
            "average_precision_mean"
        ],
        marker="*",
        s=180,
        color="#C00000",
        zorder=3,
    )

    parameter = method_summary[
        "parameter"
    ].iloc[0]

    if parameter == "beta":
        axis.set_xscale(
            "log"
        )

    axis.set_title(
        method_id.upper().replace(
            "_",
            "-",
        )
    )

    axis.set_xlabel(
        parameter
    )

    axis.set_ylabel(
        "Mean average precision"
    )

    axis.grid(
        alpha=0.25,
    )

figure.tight_layout()

figure.savefig(
    figure_directory
    / "parameter_sensitivity_average_precision.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "parameter_sensitivity_average_precision.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
figure, axes = plt.subplots(
    2,
    3,
    figsize=(
        16,
        9,
    ),
)

for method_id, axis in zip(
    method_ids,
    axes.ravel(),
    strict=True,
):
    method_summary = (
        network_summary[
            network_summary[
                "method_id"
            ]
            == method_id
        ]
    )

    matrix = (
        method_summary
        .pivot(
            index="network",
            columns="parameter_value",
            values="average_precision_mean",
        )
        .sort_index(
            axis=0
        )
        .sort_index(
            axis=1
        )
    )

    image = axis.imshow(
        matrix.to_numpy(),
        aspect="auto",
        cmap="Blues",
        vmin=0.0,
        vmax=1.0,
    )

    axis.set_xticks(
        range(
            len(
                matrix.columns
            )
        )
    )

    axis.set_xticklabels(
        [
            f"{value:g}"
            for value
            in matrix.columns
        ]
    )

    axis.set_yticks(
        range(
            len(
                matrix.index
            )
        )
    )

    axis.set_yticklabels(
        matrix.index
    )

    axis.set_title(
        method_id.upper().replace(
            "_",
            "-",
        )
    )

    axis.set_xlabel(
        method_summary[
            "parameter"
        ].iloc[0]
    )

    axis.set_ylabel(
        "Network"
    )

figure.colorbar(
    image,
    ax=axes.ravel().tolist(),
    label="Mean average precision",
    fraction=0.025,
    pad=0.02,
)

figure.subplots_adjust(
    left=0.12,
    right=0.9,
    bottom=0.1,
    top=0.93,
    wspace=0.45,
    hspace=0.4,
)

figure.savefig(
    figure_directory
    / "network_parameter_sensitivity_heatmaps.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "network_parameter_sensitivity_heatmaps.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
network_summary[
    [
        "network",
        "domain",
        "method_id",
        "method",
        "parameter",
        "parameter_value",
        "is_primary",
        "average_precision_mean",
        "average_precision_std",
        "roc_auc_mean",
        "roc_auc_std",
    ]
].sort_values(
    [
        "method_id",
        "network",
        "parameter_value",
    ]
).reset_index(
    drop=True
)